In [6]:
 

from typing import Annotated, List, TypedDict, Literal
# from typing_extensions import TypedDict
from langgraph.graph import START, END, MessagesState, StateGraph
from langgraph.graph.message import add_messages
# from langgraph.graph.state import StateGraph
from langgraph.prebuilt import ToolNode
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage
import os
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool



In [7]:
from dotenv import load_dotenv

load_dotenv()

import os

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
# os.environ['LANGCHAIN_API_KEY']=os.getenv('LANGCHAIN_API_KEY')
# os.environ['LANGSMITH_TRACING_V2']="true"
# os.environ['LANGSMITH_PROJECT']="langsmithlearning"
# os.environ['LANGCHAIN_ENDPOINT']="https://api.smith.langchain.com"

In [8]:
class AgentState(MessagesState):
    next_agent:str #Which agent should go next
    

In [10]:
## Create simple tool
@tool
def search_web(query:str) -> str:
    """Search the web for information"""
    search = TavilySearchResults(max_results=3)
    results = search.invoke(query)
    return str(results)

@tool
def write_summary(content:str) -> str:
    """Write a summary of the content"""
    summary = f"Summary of the findings: {content[:500]}"
    return summary


In [11]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("groq:llama-3.3-70b-versatile")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000015EABF1C150>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000015EAC068E50>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [12]:
## define the agent functions (simpler approach)
def researcher_agent(state:AgentState):
    """"Researcher agent that searches for infromation"""
    messages = state["messages"]

    ## add system message for context
    system_msg = SystemMessage(content="You are a research assistant. Use the search web tool to find the information about the user's request.")

    ## call the llm with tools
    researcher_llm = llm.bind_tools([search_web])
    response = researcher_llm.invoke([system_msg] + messages)

    ## return the response and route to the writer
    return {
        "messages": [response],
        "next_agent": "writer"
    } 

In [13]:
def writer_agent(state:AgentState):
    """Writer agent that writes a summary"""
    messages = state["messages"]

    ## Add system message for context
    system_msg = SystemMessage(content="You are a technical writer. Read the conversation and write a summary.")

    ## Simple completion without tools
    response = llm.invoke([system_msg] + messages)

    return {
        "messages": [response],
        "next_agent": "end"
    }

In [14]:
## tool executer nodes
def execute_tools(state:AgentState):
    """Executes any pending tool calls"""
    messages = state["messages"]
    last_message = messages[-1]

    ## check if there are tool calls to execute
    if hasattr(last_message, "tool_call") and last_message.tool_call:
        # create tool nodea and execute tool
        tool_node = ToolNode([search_web, write_summary])
        response = tool_node.invoke(state)

        return response

    # no tool to execute
    return state

In [ ]:
# build graph
workflow = StateGraph(MessagesState)

# Add nodes
workflow.add_node(START, researcher_agent)
workflow.add_node("writer", writer_agent)

# define flow
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "end")
final_workflow = workflow.compile()
print(final_workflow)
# return workflow.compile()


In [ ]:
# ## Minimal working example with real LLM
# def create_minimal_multi_agent():
#     """Minimal multi agent systaem using just LLM calls"""
#     # def researcher_node(state: MessagesState):
#     #     """Simple researcher agent that searches for infromation"""
#     #     messages = state["messages"]

#     #     ## add system message for context
#     #     prompt = [ SystemMessage(content="You are a research assistant. Use the search web tool to find the information about the user's request."),
#     #               *messages
#     #               ]

#     #     #get response
#     #     response = llm.invoke(prompt)

#     #     #return response
#     #     return {
#     #         "messages": [response]
#     #     }
    
#     # def writer_node(state: MessagesState):
#     #     """Simple writer agent that writes a summary"""
#     #     messages = state["messages"]

#     #     ## add system message for context
#     #     prompt = [ SystemMessage(content="You are a technical writer. Read the conversation and write a summary."),
#     #               *messages
#     #               ]

#     #     #get response
#     #     response = llm.invoke(prompt)

#     #     #return response
#     #     return {
#     #         "messages": [response]
#     #     }
        